In [8]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("spotify_millsongdata.csv")

df.head()

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [3]:
df.columns

Index(['artist', 'song', 'link', 'text'], dtype='str')

In [9]:
df = df[['artist', 'song', 'text']]

df = df.fillna('')

df.head()

,artist,song,text
0,ABBA,Ahe's My Kind Of Girl,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante","Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,Making somebody happy is a question of give an...


In [11]:
df['features'] = (
    df['song'] + ' ' +
    df['artist'] + ' ' +
    df['text']
)

df.head()

,artist,song,text,features
0,ABBA,Ahe's My Kind Of Girl,"Look at her face, it's a wonderful face \r\nA...","Ahe's My Kind Of Girl ABBA Look at her face, i..."
1,ABBA,"Andante, Andante","Take it easy with me, please \r\nTouch me gen...","Andante, Andante ABBA Take it easy with me, pl..."
2,ABBA,As Good As New,I'll never know why I had to go \r\nWhy I had...,As Good As New ABBA I'll never know why I had ...
3,ABBA,Bang,Making somebody happy is a question of give an...,Bang ABBA Making somebody happy is a question ...
4,ABBA,Bang-A-Boomerang,Making somebody happy is a question of give an...,Bang-A-Boomerang ABBA Making somebody happy is...


In [12]:
vectorizer = CountVectorizer(
    stop_words='english',
    min_df=20
)

word_matrix = vectorizer.fit_transform(df['features'])

print(word_matrix.shape)

(57650, 10294)


In [13]:
def get_recommendations(song_name, df, word_matrix, count=10):

    # Song find karo
    index = df.index[
        df['song'].str.lower() == song_name.lower()
    ]

    if len(index) == 0:
        return []

    idx = index[0]

    # Sirf selected song ki similarity calculate karo
    similarities = cosine_similarity(
        word_matrix[idx],
        word_matrix
    ).flatten()

    recommendations = list(enumerate(similarities))

    recommendations = sorted(
        recommendations,
        key=lambda x: x[1],
        reverse=True
    )

    top_recs = recommendations[1:count+1]

    songs = []

    for rec in top_recs:

        title = df.iloc[rec[0]]['song']
        artist = df.iloc[rec[0]]['artist']

        songs.append(
            f"{title} - {artist}"
        )

    return songs

In [14]:
df['song'].sample(20)

23321                        You Won't Be There
23261                      Children Of The Moon
52233    Met A Little Girl On Her Way To School
40359                                 The Crypt
49964                                   Vampire
13053                               Ready 2 Win
16782                        Down Is The New Up
38877                                These Days
43241                                The Circus
3574                          Hat Full Of Stars
9668                         A Little At A Time
11632                     It's Too Soon To Know
31472                               Get Up John
33980                          Poor Happy Jimmy
34579                                 Hound Dog
14930                                      Stay
48149                              Equal Rights
28246                                  Bad Time
12609                    Rejoice With Trembling
53433                              Spanish Eyes
Name: song, dtype: str

In [16]:
results = get_recommendations(
    "Hello",
    df,
    word_matrix
)

for song in results:
    print(song)

Pop! Goes The Weasel - Unknown
Song For Him - Bob Seger
Hello Hello - Elton John
Hello Goodbye - Glee
Hello Again - Neil Diamond
Butterflies - Sia
This Part Of Town - Widespread Panic
Can't Let Go - Out Of Eden
If You See Her Say Hello - Bob Dylan
Hello Again - Regine Velasquez
